# Historical Weather - Bulk Only (Radius Average)

This notebook reads NOAA GHCN daily bulk files and returns one row per:
- location
- day
- datatype found in the data

For each row, it averages all station measurements within a configurable radius.
No NOAA API calls are used in this notebook.

## Datatype Documentation

### Precipitation & Snow
- `PRCP`: Precipitation (measured in tenths of mm). Includes all forms of water (rain, melted snow/sleet) during the reporting period.
- `SNOW`: Snowfall (measured in mm). Amount of new snow that fell.
- `SNWD`: Snow depth (measured in mm). Total depth of snow on the ground.
- `WESD`: Water equivalent of snow on the ground (tenths of mm). Snowpack liquid-water equivalent.
- `WESF`: Water equivalent of snowfall (tenths of mm). Liquid-water equivalent of newly fallen snow.

### Temperature Metrics
- `TMAX`: Maximum temperature (stored as tenths of degrees Celsius in NOAA). Converted to degrees Celsius in this notebook.
- `TMIN`: Minimum temperature (stored as tenths of degrees Celsius in NOAA). Converted to degrees Celsius in this notebook.
- `TOBS`: Temperature at the time of observation (stored as tenths of degrees Celsius in NOAA). Converted to degrees Celsius in this notebook.

### Data Quality & Reporting Period
- `DAPR`: Number of days included in a multiday precipitation total.
- `MDPR`: Multiday precipitation total. Use with `DAPR` to estimate average daily precipitation for reporting gaps.

In [41]:
import csv
import gzip
import math
import tempfile
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd
import requests

In [ ]:
# -----------------------------
# CONFIGURATION
# -----------------------------
# Date range to pull from NOAA bulk files.
# By default this is "today back to 3 years ago".
_today = date.today()
START_DATE = (_today - timedelta(days=10)).isoformat()
# START_DATE = _today.replace(year=_today.year - 3).isoformat()
END_DATE = _today.isoformat()

# Radius in kilometers used to choose contributing stations.
# Every station inside this radius is eligible to contribute
# to the average for a given day/datatype.
RADIUS_KM = 20

# One or more target locations.
# Add or remove entries as needed.
LOCATIONS = [
    {"id": "nike-portland", "latitude": 45.5204, "longitude": -122.6782},
    {"id": "nike-by-lake-oswego", "latitude": 45.4124, "longitude": -122.7303},
    {"id": "nike-sydney-city", "latitude": -33.8642, "longitude": 151.2066}
]

# Datatypes to always include in output rows, even if missing in the selected date range.
EXPECTED_DATATYPES = ["PRCP", "SNOW", "SNWD", "WESD", "WESF", "TMAX", "TMIN", "TOBS", "DAPR", "MDPR"]

# Set to True to clear downloaded NOAA cache files before running.
CLEAR_CACHE = True

# Output CSV path.
OUTPUT_CSV = "bulk_weather_output.csv"

In [43]:
# -----------------------------
# CORE IMPLEMENTATION
# -----------------------------
# This cell contains all helper functions needed for the notebook.
# It is intentionally self-contained so you only need to run:
# 1) imports cell
# 2) config cell
# 3) this cell
# 4) run cell

# Local cache folder for large NOAA files so repeat runs are faster.
_BULK_CACHE = Path(tempfile.gettempdir()) / "noaa_ghcnd_cache"
_STATIONS_CATALOG_PATH = _BULK_CACHE / "ghcnd-stations.txt"


def to_date(value):
    """Normalize either ISO string or date object to date."""
    if isinstance(value, date):
        return value
    return datetime.fromisoformat(value).date()


def iter_days(start, end):
    """Yield every day from start to end inclusive."""
    cur = start
    while cur <= end:
        yield cur
        cur += timedelta(days=1)


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in kilometers between two lat/lon points."""
    r = 6371.0088
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return r * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def normalize_noaa_value(datatype, value):
    """Convert NOAA raw value to analysis-friendly units for supported datatypes."""
    if datatype in {"TMAX", "TMIN", "TOBS"}:
        # NOAA stores temperature as tenths of degrees C.
        return value / 10.0
    return value


def unit_hint(datatype):
    """Human-readable unit guidance for common datatypes after normalization."""
    return {
        "PRCP": "tenths of mm",
        "SNOW": "mm",
        "SNWD": "mm",
        "WESD": "tenths of mm",
        "WESF": "tenths of mm",
        "TMAX": "degrees C (converted from tenths of degrees C)",
        "TMIN": "degrees C (converted from tenths of degrees C)",
        "TOBS": "degrees C (converted from tenths of degrees C)",
        "DAPR": "days",
        "MDPR": "tenths of mm",
    }.get(datatype, "NOAA raw units (datatype-dependent)")


def clear_bulk_cache():
    """Remove cached NOAA bulk/year and station files when a fresh pull is needed."""
    if not _BULK_CACHE.exists():
        return
    removed = 0
    for p in _BULK_CACHE.glob("*.csv.gz"):
        try:
            p.unlink()
            removed += 1
        except OSError:
            pass
    if _STATIONS_CATALOG_PATH.exists():
        try:
            _STATIONS_CATALOG_PATH.unlink()
            removed += 1
        except OSError:
            pass
    print(f"Cleared NOAA cache artifacts: {removed}")


def ensure_bulk_csv(year):
    """
    Download yearly bulk CSV gzip once and reuse it from cache.
    """
    _BULK_CACHE.mkdir(parents=True, exist_ok=True)
    path = _BULK_CACHE / f"{year}.csv.gz"
    if not (path.exists() and path.stat().st_size > 0):
        url = f"https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/{year}.csv.gz"
        print(f"Downloading bulk file for {year}...")
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        path.write_bytes(resp.content)
    return path


def ensure_stations_catalog():
    """
    Download station catalog once and reuse it from cache.
    """
    _BULK_CACHE.mkdir(parents=True, exist_ok=True)
    if not (_STATIONS_CATALOG_PATH.exists() and _STATIONS_CATALOG_PATH.stat().st_size > 0):
        url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt"
        print("Downloading station catalog...")
        resp = requests.get(url, timeout=180)
        resp.raise_for_status()
        _STATIONS_CATALOG_PATH.write_bytes(resp.content)
    return _STATIONS_CATALOG_PATH


def iter_station_catalog_rows(path):
    """
    Parse fixed-width station catalog rows.
    Yields: station_id, station_name, lat, lon
    """
    with path.open("r", encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            if len(line) < 30:
                continue
            sid = line[0:11].strip()
            try:
                lat = float(line[12:20])
                lon = float(line[21:30])
            except ValueError:
                continue
            name = line[41:71].strip() if len(line) >= 71 else sid
            yield sid, name, lat, lon


def build_bulk_radius_candidates(locations, radius_km):
    """
    For each location, build a list of all stations within radius_km.
    The list is sorted by distance ascending.
    """
    catalog_path = ensure_stations_catalog()
    out = {}

    for loc in locations:
        loc_id = str(loc.get("id", loc))
        llat, llon = float(loc["latitude"]), float(loc["longitude"])
        candidates = []

        for sid, name, lat, lon in iter_station_catalog_rows(catalog_path):
            dist = haversine_km(llat, llon, lat, lon)
            if dist <= radius_km:
                candidates.append(
                    {
                        "station_id": f"GHCND:{sid}",
                        "station_name": name,
                        "station_latitude": lat,
                        "station_longitude": lon,
                        "distance_km": dist,
                    }
                )

        candidates.sort(key=lambda x: x["distance_km"])
        out[loc_id] = candidates

    return out


def collect_bulk_obs_for_candidate_union(candidate_by_loc, start, end):
    """
    Scan yearly CSV files once and keep only rows for stations inside
    any location radius. Output structure:
      obs_index[(station_id, date)] = [observation, observation, ...]
    """
    union_station_bare = {
        c["station_id"].split(":", 1)[1]
        for rows in candidate_by_loc.values()
        for c in rows
    }

    obs_index = defaultdict(list)

    for year in range(start.year, end.year + 1):
        csv_path = ensure_bulk_csv(year)
        with gzip.open(csv_path, "rt", encoding="utf-8") as fh:
            for row in csv.reader(fh):
                if len(row) < 4:
                    continue

                bare_sid = row[0]
                if bare_sid not in union_station_bare:
                    continue

                datatype = row[2]

                try:
                    obs_date = datetime.strptime(row[1], "%Y%m%d").date()
                except ValueError:
                    continue

                if obs_date < start or obs_date > end:
                    continue

                m, q, s, t = (row[i] if len(row) > i else "" for i in (4, 5, 6, 7))
                attrs = f"M={m};Q={q};S={s};T={t}".strip(";") or None
                full_sid = f"GHCND:{bare_sid}"
                raw_value = float(row[3])

                obs_index[(full_sid, obs_date)].append(
                    {
                        "station_id": full_sid,
                        "date": obs_date,
                        "datatype": datatype,
                        "value": normalize_noaa_value(datatype, raw_value),
                        "raw_unit_hint": unit_hint(datatype),
                        "attributes": attrs,
                    }
                )

    return obs_index


def fetch_weather_dataframe_bulk_only(locations, start_date, end_date, radius_km=50, expected_datatypes=None):
    """
    Main pipeline:
    1) Build stations-in-radius for each location
    2) Scan bulk NOAA data once for those stations
    3) For each location/day/datatype, average same-unit observations
    4) Return a flat dataframe with contributor details

    Important behavior:
    - No datatype filter is applied.
    - Datatypes are discovered from the data itself.
    - Optional expected_datatypes are always included in output rows.
    - Missing datatype for a day returns a row with value=None.
    """
    start, end = to_date(start_date), to_date(end_date)
    total_days = (end - start).days + 1

    print(f"Building station candidates within {radius_km} km radius per location...")
    candidates_by_loc = build_bulk_radius_candidates(locations, radius_km=radius_km)

    for loc_id, candidates in candidates_by_loc.items():
        print(f"  [{loc_id}] {len(candidates)} stations within {radius_km} km")

    print("Scanning NOAA bulk files for candidate stations...")
    obs_index = collect_bulk_obs_for_candidate_union(candidates_by_loc, start, end)

    # Quick lookup for station metadata by location + station id.
    station_info_by_loc = {
        loc_id: {c["station_id"]: c for c in candidates}
        for loc_id, candidates in candidates_by_loc.items()
    }

    # Build complete datatype set from observed data union expected datatypes.
    observed_datatypes = {
        obs["datatype"]
        for obs_list in obs_index.values()
        for obs in obs_list
    }
    expected_datatypes = set(expected_datatypes or [])
    target_datatypes = sorted(observed_datatypes | expected_datatypes)

    print(
        f"Datatypes in output: {len(target_datatypes)} total "
        f"({len(observed_datatypes)} observed + {len(expected_datatypes)} expected-configured)"
    )

    records = []

    # Build one output row per location/day/datatype.
    for loc in locations:
        location_id = str(loc.get("id", loc))
        lat, lon = float(loc["latitude"]), float(loc["longitude"])
        candidates = candidates_by_loc[location_id]
        station_info = station_info_by_loc[location_id]
        candidate_sids = {c["station_id"] for c in candidates}

        print(f"[{location_id}] Averaging stations within {radius_km} km ({total_days:,} days)...")

        for day in iter_days(start, end):
            # Group all observations for this day by datatype.
            day_obs_by_type = defaultdict(list)
            for (full_sid, obs_date), obs_list in obs_index.items():
                if obs_date != day or full_sid not in candidate_sids:
                    continue
                for obs in obs_list:
                    day_obs_by_type[obs["datatype"]].append(obs)

            for dtype in target_datatypes:
                obs_list = day_obs_by_type.get(dtype, [])

                if not obs_list:
                    # Preserve row completeness even when data is missing.
                    records.append(
                        {
                            "location_id": location_id,
                            "latitude": lat,
                            "longitude": lon,
                            "date": day.isoformat(),
                            "datatype": dtype,
                            "value": None,
                            "station_count": 0,
                            "contributing_stations": [],
                            "raw_unit_hint": unit_hint(dtype),
                        }
                    )
                    continue

                # Safety: only average observations that share the same unit hint.
                # If mixed units appear, we keep the dominant group by frequency.
                by_unit = defaultdict(list)
                for obs in obs_list:
                    by_unit[obs["raw_unit_hint"]].append(obs)

                unit_hint_to_use = max(by_unit.keys(), key=lambda u: len(by_unit[u]))
                obs_same_unit = by_unit[unit_hint_to_use]

                # Keep full station details used in the average.
                contributing_stations = []
                for obs in obs_same_unit:
                    sid = obs["station_id"]
                    station_info_rec = station_info[sid]
                    contributing_stations.append(
                        {
                            "station_id": sid,
                            "station_name": station_info_rec["station_name"],
                            "station_latitude": station_info_rec["station_latitude"],
                            "station_longitude": station_info_rec["station_longitude"],
                            "distance_km": round(station_info_rec["distance_km"], 6),
                            "value": obs["value"],
                        }
                    )

                contributing_stations.sort(key=lambda x: x["distance_km"])

                values = [obs["value"] for obs in obs_same_unit]
                avg_value = sum(values) / len(values)

                records.append(
                    {
                        "location_id": location_id,
                        "latitude": lat,
                        "longitude": lon,
                        "date": day.isoformat(),
                        "datatype": dtype,
                        "value": avg_value,
                        "station_count": len(values),
                        "contributing_stations": contributing_stations,
                        "raw_unit_hint": unit_hint_to_use,
                    }
                )

    return pd.DataFrame(records)

In [44]:
# -----------------------------
# RUN EXTRACTION
# -----------------------------
# This executes the full bulk pull with no datatype filter.
# Datatypes are auto-discovered from the data in the selected date range.
if CLEAR_CACHE:
    clear_bulk_cache()

df_bulk = fetch_weather_dataframe_bulk_only(
    locations=LOCATIONS,
    start_date=START_DATE,
    end_date=END_DATE,
    radius_km=RADIUS_KM,
    expected_datatypes=EXPECTED_DATATYPES,
)

print(f"Bulk rows: {len(df_bulk):,}")

# Save to CSV. If the file is open/locked, write to a timestamped fallback name.
if OUTPUT_CSV:
    try:
        df_bulk.to_csv(OUTPUT_CSV, index=False)
        print(f"Saved {OUTPUT_CSV}")
    except PermissionError:
        fallback_csv = f"bulk_weather_output_{datetime.now().strftime('%Y%m%d_%H%M%S')}" + ".csv"
        df_bulk.to_csv(fallback_csv, index=False)
        print(f"Primary file was locked. Saved {fallback_csv} instead.")

# Show result preview.
df_bulk

Cleared NOAA cache artifacts: 2
Building station candidates within 20 km radius per location...
  [nike-portland] 171 stations within 20 km
  [nike-by-lake-oswego] 166 stations within 20 km
  [nike-sydney-city] 145 stations within 20 km
Scanning NOAA bulk files for candidate stations...
Datatypes in output: 15 total (15 observed + 10 expected-configured)
[nike-portland] Averaging stations within 20 km (6 days)...
[nike-by-lake-oswego] Averaging stations within 20 km (6 days)...
[nike-sydney-city] Averaging stations within 20 km (6 days)...
Bulk rows: 270
Saved bulk_weather_output.csv


,location_id,latitude,longitude,date,datatype,value,station_count,contributing_stations,raw_unit_hint
0,nike-portland,45.5204,-122.6782,2026-04-28,AWND,22.0,2,"[{'station_id': 'GHCND:USW00024229', 'station_...",NOAA raw units (datatype-dependent)
1,nike-portland,45.5204,-122.6782,2026-04-28,DAPR,NaN,0,[],days
2,nike-portland,45.5204,-122.6782,2026-04-28,MDPR,NaN,0,[],tenths of mm
3,nike-portland,45.5204,-122.6782,2026-04-28,PRCP,0.0,48,"[{'station_id': 'GHCND:USC00356749', 'station_...",tenths of mm
4,nike-portland,45.5204,-122.6782,2026-04-28,SNOW,0.0,44,"[{'station_id': 'GHCND:US1ORMT0033', 'station_...",mm
...,...,...,...,...,...,...,...,...,...
265,nike-sydney-city,-33.8642,151.2066,2026-05-03,WDF5,NaN,0,[],NOAA raw units (datatype-dependent)
266,nike-sydney-city,-33.8642,151.2066,2026-05-03,WESD,NaN,0,[],tenths of mm
267,nike-sydney-city,-33.8642,151.2066,2026-05-03,WESF,NaN,0,[],tenths of mm
268,nike-sydney-city,-33.8642,151.2066,2026-05-03,WSF2,NaN,0,[],NOAA raw units (datatype-dependent)
